# CSE498R baseline — Step 6: environment, T4 checks, ARC-Easy cache

Runtime: **GPU, T4**. Everything this notebook writes goes to
`/content/drive/MyDrive/qAIR_CSE498R/` (`cache/arc_easy/`, `ckpt/`, `runs/`,
`exports/`); it never reads or writes `qAIR_V45` or any older folder. Each
Colab session gets its own folder `runs/<UTC time>_<commit>/` with every log
and report.

| Cell | What it does | Stop afterwards? |
|---|---|---|
| 0 | Settings, folder guard, helpers | |
| 1 | Mount Drive, clone at `PINNED_COMMIT`, check the clone is clean | |
| 2 | Environment before installing: torch, CUDA, GPU, full `pip freeze` | |
| 3 | Remove diffusers / gradio / jax, install the pins (torch kept), `pip check`, freeze hash | **STOP 1** on the first pass: paste the summary to Claude Code, which commits the torch pin |
| **A** 4 | Smoke timing: 20 train + 10 validation questions in `runs/<session>/smoke/` | |
| A 5 | E10: two same-seed 1-epoch runs, per-example prediction diff | stops if strict determinism raises |
| A 6 | E10: `lightning.gpu` repeatability and difference from `lightning.qubit` | |
| A 7 | Section A summary | **STOP 2**: paste it to Claude Code |
| **B** 8 | Full ARC-Easy cache via `scripts/build_cache.py`: train + validation, then test (resumable) | |
| B 9 | Cache-quality gate on the full **train and validation** splits | |
| B 10 | Cache-build manifest (E6) in `runs/<session>/` | |

**Order.** First pass: cells 0–3 only, then STOP 1. After Claude Code has
committed the torch pin: set `PINNED_COMMIT` to the new commit, *Runtime →
Restart session*, run 0–3 again, then Section A, then STOP 2. Section B runs
only after Claude Code has seen Section A: set `SECTION_B_APPROVED = True`
in cell 0 and run 0–3, then 8–10 (skip Section A).

**If Colab disconnects during cell 8:** run 0–3 again, then cell 8 again; the
build resumes from its last autosave (every 50 questions). If it then says
another build is running, that is the dead build's lock: wait until its last
autosave is 15 minutes old and run cell 8 again. Do not delete the lock.

The test split is built but never gated: its cache-quality block is hidden in
cell 8 and kept only in `build_test.log`; do not use those numbers for any
decision (CLAUDE.md rule 5).

## 0 · Settings and helpers

In [ ]:
# ---- Settings: edit only these ----------------------------------------
PINNED_COMMIT = "REPLACE_WITH_THE_COMMIT_IN_docs/HANDOFF.md"
SECTION_B_APPROVED = False   # True only after Claude Code has seen Section A
GEN_SEED = 0                 # the fixed generation seed, config.GEN_SEED (checked in cell 1)
ARC_CONFIG = "ARC-Easy"
# ------------------------------------------------------------------------

REPO_URL = "https://github.com/Thorfast191/CSE498R-qAIR.git"
REPO = "/content/CSE498R-qAIR"                  # code, on the local disk
ROOT = "/content/drive/MyDrive/qAIR_CSE498R"    # everything this notebook writes
CACHE = f"{ROOT}/cache/arc_easy"
CKPT = f"{ROOT}/ckpt"
RUNS = f"{ROOT}/runs"
EXPORTS = f"{ROOT}/exports"

import hashlib
import json
import os
import re
import subprocess
import sys
import time

# Never read or write an older project folder (qAIR_V45, qAIR_V44, ...).
for path in (ROOT, CACHE, CKPT, RUNS, EXPORTS):
    if re.search(r"qair_v\d", path, re.IGNORECASE):
        raise RuntimeError(f"STOP: {path} names an old project folder.")

if not re.fullmatch(r"[0-9a-f]{7,40}", PINNED_COMMIT):
    raise RuntimeError("STOP: set PINNED_COMMIT to the commit given in docs/HANDOFF.md.")


def run(cmd, log=None, hide_quality_block=False, check=True):
    """
    Runs cmd in the repo, streams its output live (progress bars included)
    and appends it to log. hide_quality_block keeps [CACHE QUALITY ...]
    blocks out of the cell output; they still go to the log.
    """
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen([str(c) for c in cmd], env=env,
                            cwd=REPO if os.path.isdir(REPO) else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    sink = open(log, "a") if log else None
    pending, in_block = b"", False

    def emit(segment):
        nonlocal in_block
        text = segment.decode("utf-8", errors="replace")
        if sink:
            sink.write(text)
            sink.flush()
        if hide_quality_block:
            if text.startswith("[CACHE QUALITY"):
                in_block = True
                sys.stdout.write("[cache-quality block hidden; not used for any decision]\n")
                return
            if in_block and (text.startswith(" ") or not text.strip()):
                return
            in_block = False
        sys.stdout.write(text)
        sys.stdout.flush()

    while True:
        chunk = proc.stdout.read1(1 << 16)
        if not chunk:
            break
        pending += chunk
        while True:
            cuts = [i for i in (pending.find(b"\n"), pending.find(b"\r")) if i >= 0]
            if not cuts:
                break
            emit(pending[:min(cuts) + 1])
            pending = pending[min(cuts) + 1:]
    if pending:
        emit(pending)

    code = proc.wait()
    if sink:
        sink.close()
    if check and code != 0:
        raise RuntimeError(f"STOP: {' '.join(map(str, cmd))} exited with {code}"
                           + (f"; full output in {log}" if log else ""))
    return code


def output(cmd):
    """stdout of cmd run in the repo."""
    return subprocess.run([str(c) for c in cmd], cwd=REPO, capture_output=True,
                          text=True, check=True).stdout.strip()


def sha256_text(text):
    return hashlib.sha256(text.encode()).hexdigest()


print("settings ok; PINNED_COMMIT =", PINNED_COMMIT,
      "; SECTION_B_APPROVED =", SECTION_B_APPROVED)

## 1 · Drive and code at the pinned commit

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for path in (CACHE, CKPT, RUNS, EXPORTS):
    os.makedirs(path, exist_ok=True)

if not os.path.isdir(REPO):
    run(["git", "clone", "--quiet", REPO_URL, REPO])
run(["git", "fetch", "--quiet", "origin"])
run(["git", "checkout", "--quiet", "--detach", PINNED_COMMIT])

COMMIT = output(["git", "rev-parse", "HEAD"])
if not COMMIT.startswith(PINNED_COMMIT):
    raise RuntimeError(f"STOP: HEAD is {COMMIT}, not {PINNED_COMMIT}.")
if output(["git", "status", "--porcelain", "--untracked-files=no"]):
    raise RuntimeError("STOP: the clone has uncommitted changes; every run would be marked dirty.")

config_seed = int(output([sys.executable, "-c", "import config; print(config.GEN_SEED)"]))
if config_seed != GEN_SEED:
    raise RuntimeError(f"STOP: GEN_SEED is {GEN_SEED} here but config.GEN_SEED is {config_seed}.")

SESSION = f"{RUNS}/{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}_{COMMIT[:7]}"
os.makedirs(SESSION)

print("commit :", COMMIT)
print("session:", SESSION)

## 2 · Environment before installing anything (E11)

In [ ]:
ENV_REPORT = """
import torch
print("torch             ", torch.__version__)
print("torch.version.cuda", torch.version.cuda)
print("cuda available    ", torch.cuda.is_available())
print("gpu               ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
"""
run([sys.executable, "-c", ENV_REPORT], log=f"{SESSION}/env_before.txt")

freeze_before = output([sys.executable, "-m", "pip", "freeze"])
with open(f"{SESSION}/env_before.txt", "a") as f:
    f.write("\n# pip freeze\n" + freeze_before + "\n")
print("\n# pip freeze (before install)\n" + freeze_before)

TORCH = output([sys.executable, "-c", "import torch; print(torch.__version__)"])
CUDA = output([sys.executable, "-c", "import torch; print(torch.version.cuda)"])
GPU = output([sys.executable, "-c", "import torch; print(torch.cuda.get_device_name(0) "
              "if torch.cuda.is_available() else 'none')"])

if GPU == "none":
    raise RuntimeError("STOP: no GPU. Runtime -> Change runtime type -> T4 GPU.")
if "T4" not in GPU:
    print(f"\n[WARNING] GPU is {GPU}, not a T4; timings will not match the T4 plan.")

## 3 · Clean up, install the pins, check

Removes the packages that conflicted in the v46 run (huggingface-hub vs
diffusers / gradio; PennyLane vs JAX), installs `requirements.txt` with torch
held at Colab's version (torch is never reinstalled), adds
`pennylane-lightning[gpu]`, then runs `pip check`.

In [ ]:
INSTALL_LOG = f"{SESSION}/install.log"
run([sys.executable, "-m", "pip", "uninstall", "-y", "diffusers", "gradio", "jax", "jaxlib"],
    log=INSTALL_LOG)

with open(f"{REPO}/requirements.txt") as f:
    requirements = f.read().splitlines()

torch_line = re.compile(r"\s*torch\s*(==|$)")
torch_pin = next((line.split("==", 1)[1].strip() for line in requirements
                  if torch_line.match(line) and "==" in line), None)
if torch_pin is not None and torch_pin != TORCH:
    raise RuntimeError(f"STOP: requirements.txt pins torch=={torch_pin}, Colab ships {TORCH}. "
                       f"Do not reinstall torch; tell Claude Code.")

without_torch = f"{SESSION}/requirements_without_torch.txt"
with open(without_torch, "w") as f:
    f.write("\n".join(line for line in requirements if not torch_line.match(line)) + "\n")
keep_torch = f"{SESSION}/keep_torch.txt"
with open(keep_torch, "w") as f:
    f.write(f"torch=={TORCH}\n")

run([sys.executable, "-m", "pip", "install", "-q", "-r", without_torch, "-c", keep_torch,
     "pennylane-lightning[gpu]==0.42.0"], log=INSTALL_LOG)

torch_after = output([sys.executable, "-c", "import torch; print(torch.__version__)"])
if torch_after != TORCH:
    raise RuntimeError(f"STOP: torch changed from {TORCH} to {torch_after}. "
                       f"Runtime -> Disconnect and delete runtime, then tell Claude Code.")

pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
PIP_CHECK_OK = pip_check.returncode == 0
freeze_after = output([sys.executable, "-m", "pip", "freeze"])
with open(f"{SESSION}/env_after.txt", "w") as f:
    f.write("# pip check\n" + pip_check.stdout + pip_check.stderr
            + "\n# pip freeze\n" + freeze_after + "\n")

FREEZE_SHA = sha256_text(freeze_after)
PACKAGES_SHA = output([sys.executable, "-c", "from training.environment import "
                       "environment_record; print(environment_record()[1]['packages_sha256'])"])

print("# pip check\n" + (pip_check.stdout + pip_check.stderr).strip())
print("\n" + "=" * 66)
print("ENVIRONMENT (paste to Claude Code)")
print("=" * 66)
print("commit            ", COMMIT)
print("torch             ", TORCH)
print("torch.version.cuda", CUDA)
print("gpu               ", GPU)
print("pip check ok      ", PIP_CHECK_OK)
print("pip freeze sha256 ", FREEZE_SHA)
print("packages_sha256   ", PACKAGES_SHA, " (training/environment.py; the manifests record this one)")
print("full lists        ", f"{SESSION}/env_before.txt, {SESSION}/env_after.txt")
if torch_pin is None:
    print("\nSTOP 1: torch is not pinned yet. Paste the block above to Claude Code; it commits "
          "torch==<version> and gives you the new PINNED_COMMIT.")
else:
    print(f"\ntorch pinned ({torch_pin}) and installed; continue.")

# Section A — measurements on a small smoke cache

Run only at the commit with the torch pin. Nothing here touches
`cache/arc_easy/`.

## 4 · Smoke timing (E8)

In [ ]:
SMOKE = f"{SESSION}/smoke"
run([sys.executable, "scripts/smoke_timing.py",
     "--cache-dir", f"{SMOKE}/cache", "--ckpt-dir", f"{SMOKE}/ckpt",
     "--train", 20, "--val", 10, "--arc-config", ARC_CONFIG, "--gen-seed", GEN_SEED],
    log=f"{SESSION}/smoke_timing.log")

## 5 · Determinism pair (E10)

Two same-seed 1-epoch runs on the smoke cache, each in its own folder; their
histories, best checkpoints and per-example validation predictions are
compared. Strict deterministic mode stays on: if an operation raises, the
cell stops. Report it; do not rerun with `--deterministic-warn-only`.

In [ ]:
code = run([sys.executable, "scripts/t4_checks.py", "determinism",
            "--cache-dir", f"{SMOKE}/cache", "--out-dir", f"{SESSION}/determinism",
            "--train", 20, "--val", 10, "--arc-config", ARC_CONFIG],
           log=f"{SESSION}/determinism.log", check=False)
if code == 2:
    raise RuntimeError("STOP: strict deterministic mode raised (operation printed above). "
                       "Report it to Claude Code; no retry with --deterministic-warn-only.")
if code:
    raise RuntimeError(f"STOP: the determinism check failed (exit {code}); see {SESSION}/determinism.log")

## 6 · `lightning.gpu` check (E10)

In [ ]:
code = run([sys.executable, "scripts/t4_checks.py", "quantum", "--device", "lightning.gpu",
            "--out", f"{SESSION}/quantum_check.json"],
           log=f"{SESSION}/quantum_check.log", check=False)
if code == 3:
    print("\nlightning.gpu could not be constructed (reason above); report it.")
elif code:
    raise RuntimeError(f"STOP: the quantum check failed (exit {code}); see {SESSION}/quantum_check.log")

## 7 · Section A summary — STOP 2

In [ ]:
def load(name):
    path = f"{SESSION}/{name}"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


determinism = load("determinism/determinism.json")
summary = {
    "commit": COMMIT, "session": SESSION, "torch": TORCH, "cuda": CUDA, "gpu": GPU,
    "pip_check_ok": PIP_CHECK_OK, "pip_freeze_sha256": FREEZE_SHA, "packages_sha256": PACKAGES_SHA,
    "smoke_timing": load("smoke/ckpt/smoke_timing.json"),
    "determinism": None if determinism is None else
        {k: v for k, v in determinism.items() if k != "manifest"},
    "quantum_check": load("quantum_check.json"),
}
with open(f"{SESSION}/section_a_summary.json", "w") as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print(json.dumps(summary, indent=2, sort_keys=True))
print(f"\nSTOP 2: paste everything above to Claude Code (also in {SESSION}/section_a_summary.json).")
print("Run Section B only after that: set SECTION_B_APPROVED = True in cell 0.")

# Section B — full ARC-Easy cache (2251 train / 570 validation / 2376 test)

Only via `scripts/build_cache.py`, with the fixed seed. Resumable: after a
disconnect, run cells 0–3 and this cell again.

## 8 · Build the cache

In [ ]:
if not SECTION_B_APPROVED:
    raise RuntimeError("STOP: Section B runs only after Claude Code has seen Section A "
                       "(SECTION_B_APPROVED in cell 0).")

BUILD = [sys.executable, "scripts/build_cache.py", "--arc-config", ARC_CONFIG,
         "--cache-dir", CACHE, "--gen-seed", GEN_SEED]
LOCK_ADVICE = ("\nIf it says another build is running right after a disconnect, that is the dead "
               "build's lock: wait until its last autosave is 15 min old and run this cell again. "
               "Do not delete the lock.")

try:
    run(BUILD + ["--splits", "train", "validation"], log=f"{SESSION}/build_train_validation.log")
    # The test split is built but never gated: its quality block stays in the log only.
    run(BUILD + ["--splits", "test"], log=f"{SESSION}/build_test.log", hide_quality_block=True)
except RuntimeError:
    print(LOCK_ADVICE)
    raise

## 9 · Cache-quality gate (train and validation only)

Loads the finished caches read-only (an incomplete cache raises) and prints
the `[CACHE QUALITY]` block and the size of each split.

In [ ]:
GATE = """
import sys
from config import ARC_SPLIT_SIZES
from training.dataset import QAIRDataset

cache_dir, arc_config = sys.argv[1], sys.argv[2]
for split in ("train", "validation"):
    ds = QAIRDataset(split=split, cache_dir=cache_dir, arc_config=arc_config)
    expected = ARC_SPLIT_SIZES[arc_config][split]
    print(f"[SIZE {split}] {len(ds)} of {expected}" + ("" if len(ds) == expected else "  <-- MISMATCH"))
"""
run([sys.executable, "-c", GATE, CACHE, ARC_CONFIG], log=f"{SESSION}/cache_quality_gate.log")

## 10 · Cache-build manifest (E6)

In [ ]:
run([sys.executable, "scripts/cache_manifest.py", "--cache-dir", CACHE, "--arc-config", ARC_CONFIG,
     "--out", f"{SESSION}/cache_manifest.json"], log=f"{SESSION}/cache_manifest.log")

print(f"\nPaste to Claude Code: the cell 9 output and the lines above. Files: {SESSION}/")